# S6 — Zero-Shot Cross-Dataset Transfer: GigaDB → BCI Competition IV 2a

**Paper §2.9 / §3.9** | Author: 이원석 | Last updated: 2026-05-25

## What this notebook does

1. Mounts Google Drive (where pre-processed GigaDB HDF5 files and BCIC IV 2a GDF files live)
2. Installs required packages
3. Trains an `EEGNet22` model on all 52 GigaDB subjects using only the 22-channel subset common to BCIC IV 2a
4. Evaluates the trained model **zero-shot** (no fine-tuning) on all 9 BCIC IV 2a subjects (A01–A09, left/right MI only)
5. Saves `transfer_bcic2a_results.csv` and `transfer_bcic2a_summary.json` to Drive
6. Prints a formatted table suitable for copy-pasting into **§3.9** of `results_draft.md`

## Required Drive layout

```
MyDrive/
└── BCI_Research/
    ├── preprocessed/member_A/          ← GigaDB HDF5 files (s01.h5 … s52.h5)
    ├── BCICIV_2a_gdf/                  ← A01T.gdf A01E.gdf … A09T.gdf A09E.gdf
    ├── results/
    │   └── transfer_bcic2a/            ← output folder (auto-created)
    └── code/MI-BCI/src/               ← transfer_bcic2a.py lives here
```

> **Runtime:** GPU (T4 or better recommended). Training on 52 subjects takes ~15–25 min on T4.
> If `transfer_22ch_model.pt` already exists in `results/transfer_bcic2a/`, add `--skip_train` to the run command.

---
## Step 1 — Mount Google Drive

In [21]:
import os
# Only mount Drive when actually running on Colab (env var set by Colab runtime)
if 'COLAB_BACKEND_VERSION' in os.environ:
    from google.colab import drive
    drive.mount('/content/drive')
    DRIVE_ROOT = '/content/drive/MyDrive/BCI_Research'
    assert os.path.isdir(DRIVE_ROOT), f'Drive root not found: {DRIVE_ROOT}'
    print('Drive mounted. Root:', DRIVE_ROOT)
else:
    print('Not running in Colab — skipping Drive mount. Local paths will be used in Step 3.')


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Drive mounted. Root: /content/drive/MyDrive/BCI_Research


---
## Step 2 — Install dependencies

In [22]:
# Colab already has: torch, numpy, scipy, pandas, scikit-learn
# We only need MNE for reading GDF files
!pip install mne h5py --quiet
import mne
print('MNE version:', mne.__version__)

MNE version: 1.12.1


---
## Step 3 — Verify data paths

In [23]:
import os, glob

# ── Environment detection ─────────────────────────────────────────────────────
# If the local workspace physically exists → local run; otherwise → Colab
_LOCAL_WORKSPACE = '/Volumes/a3122a1/MI-BCI'
IN_COLAB = not os.path.isdir(_LOCAL_WORKSPACE)

# ── Path configuration ────────────────────────────────────────────────────────
if IN_COLAB:
    DRIVE_ROOT = '/content/drive/MyDrive/BCI_Research'
    DATA_DIR   = os.path.join(DRIVE_ROOT, 'preprocessed/member_A')
    BCIC_DIR   = os.path.join(DRIVE_ROOT, 'BCICIV_2a_gdf')
    OUT_DIR    = os.path.join(DRIVE_ROOT, 'results/transfer_bcic2a')
    CODE_DIR   = os.path.join(DRIVE_ROOT, 'code/MI-BCI/src')
else:
    WORKSPACE  = _LOCAL_WORKSPACE
    DRIVE_ROOT = os.path.join(WORKSPACE, 'BCI_Research')
    DATA_DIR   = os.path.join(DRIVE_ROOT, 'preprocessed/member_A')
    BCIC_DIR   = os.path.join(WORKSPACE, 'BCICIV_2a_gdf')
    OUT_DIR    = os.path.join(DRIVE_ROOT, 'results/transfer_bcic2a')
    CODE_DIR   = os.path.join(WORKSPACE, 'src')

os.makedirs(OUT_DIR, exist_ok=True)

h5_files  = sorted(glob.glob(os.path.join(DATA_DIR, 's*.h5')))
gdf_train = sorted(glob.glob(os.path.join(BCIC_DIR, 'A0*T.gdf')))
gdf_eval  = sorted(glob.glob(os.path.join(BCIC_DIR, 'A0*E.gdf')))

print(f'Environment             : {"Google Colab" if IN_COLAB else "Local"}')
print(f'CODE_DIR                : {CODE_DIR}')
print(f'GigaDB HDF5 files found : {len(h5_files)} (expected 52)')
print(f'BCIC train GDF files    : {len(gdf_train)} (expected 9)')
print(f'BCIC eval  GDF files    : {len(gdf_eval)}  (expected 9)')
print(f'Output dir              : {OUT_DIR}')

assert len(gdf_train) == 9,  'Missing BCIC train GDF files (A01T.gdf … A09T.gdf)'
assert len(gdf_eval)  == 9,  'Missing BCIC eval  GDF files (A01E.gdf … A09E.gdf)'
print('\n✅ All data paths verified.')


Environment             : Google Colab
CODE_DIR                : /content/drive/MyDrive/BCI_Research/code/MI-BCI/src
GigaDB HDF5 files found : 52 (expected 52)
BCIC train GDF files    : 9 (expected 9)
BCIC eval  GDF files    : 9  (expected 9)
Output dir              : /content/drive/MyDrive/BCI_Research/results/transfer_bcic2a

✅ All data paths verified.


---
## Step 4 — Add source directory to path and import script

In [24]:
import sys, os

# If running in a container (no /Volumes access), write transfer_bcic2a.py directly
# to /content/ so the kernel can find and import it.
_SCRIPT_NAME = 'transfer_bcic2a.py'
_LOCAL_SRC   = '/Volumes/a3122a1/MI-BCI/src'
_CONTAINER_DST = '/content'

if os.path.isfile(os.path.join(_LOCAL_SRC, _SCRIPT_NAME)):
    # Local run — src is directly accessible
    CODE_DIR = _LOCAL_SRC
else:
    # Container run — embed the script at /content/transfer_bcic2a.py
    CODE_DIR = _CONTAINER_DST
    _dst_file = os.path.join(_CONTAINER_DST, _SCRIPT_NAME)
    if not os.path.isfile(_dst_file):
        _script_content = r'''"""
transfer_bcic2a.py — Zero-shot Cross-Dataset Transfer: GigaDB → BCIC IV 2a
============================================================================
실험 설계:
  1. GigaDB 52명 전체 데이터로 EEGNet 22ch EEG-only 모델 학습 (all-subjects 학습)
  2. BCIC Competition IV Dataset 2a (9명, Left/Right 손 MI, A0xT.gdf + A0xE.gdf) 에
     동일 모델을 zero-shot으로 평가
  3. 채널 매핑: GigaDB 64ch 중 BCIC IV 2a와 겹치는 22개 채널만 사용
  4. 샘플링: GigaDB 512Hz 에포크의 MI 구간(0~4s, 2048 samples)만 사용;
            BCIC IV 2a(250Hz, 4s)를 scipy.signal.resample으로 512Hz 업샘플링

실행 방법 (Colab 권장):
  python transfer_bcic2a.py \
      --data_dir       BCI_Research/preprocessed/member_A \
      --bcic_dir       BCICIV_2a_gdf \
      --out_dir        BCI_Research/results/transfer_bcic2a \
      [--skip_train]   # 체크포인트가 이미 있으면 학습 건너뜀

출력:
  results/transfer_bcic2a/
  ├── transfer_22ch_model.pt          ← GigaDB all-52 학습 체크포인트
  ├── transfer_bcic2a_results.csv     ← 피험자별 accuracy, κ, ITR
  └── transfer_bcic2a_summary.json    ← 평균 ± SD
"""

import os, sys, json, time, random, argparse, warnings
from pathlib import Path

import numpy as np
import pandas as pd
import h5py
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, ConcatDataset
from scipy.signal import butter, filtfilt, resample
from sklearn.metrics import accuracy_score, cohen_kappa_score, confusion_matrix
import mne
mne.set_log_level('ERROR')

SEED = 42
random.seed(SEED); np.random.seed(SEED)
torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True

def _amp_autocast(enabled):
    try:    return torch.amp.autocast(device_type="cuda", enabled=enabled)
    except: return torch.cuda.amp.autocast(enabled=enabled)

def _amp_scaler():
    try:    return torch.amp.GradScaler("cuda")
    except: return torch.cuda.amp.GradScaler()

GIGADB_64 = [
    'Fp1','Fp2','F7','F3','Fz','F4','F8',
    'FC5','FC3','FC1','FCz','FC2','FC4','FC6',
    'T7','C5','C3','C1','Cz','C2','C4','C6','T8',
    'TP9','CP5','CP3','CP1','CPz','CP2','CP4','CP6','TP10',
    'P7','P5','P3','P1','Pz','P2','P4','P6','P8',
    'PO9','PO7','PO3','POz','PO4','PO8','PO10',
    'O1','Oz','O2',
    'Iz','Fpz',
    'AF7','AF3','AFz','AF4','AF8',
    'FT9','FT7','FT8','FT10','TP7','TP8',
]
BCIC_22 = [
    'Fz','FC3','FC1','FCz','FC2','FC4','C5','C3','C1',
    'Cz','C2','C4','C6','CP3','CP1','CPz','CP2','CP4',
    'P1','Pz','P2','Oz',
]
GIGADB_22_IDX   = [GIGADB_64.index(ch) for ch in BCIC_22]
BCIC_GDF_22_IDX = list(range(22))
BCIC_LEFT_CODE  = 769
BCIC_RIGHT_CODE = 770
FS_GIGADB   = 512
FS_BCIC     = 250
FS_TARGET   = 512
N_TIMES     = 2048
GIGADB_MI_START = 256
GIGADB_MI_END   = 2304
BCIC_EPOCH_SEC  = 4.0
BCIC_EPOCH_SAMP = int(BCIC_EPOCH_SEC * FS_BCIC)
BCIC_TMIN       = 0.0

def calc_itr(acc, n_classes=2, trial_sec=4.0):
    p = np.clip(acc, 1e-9, 1.0 - 1e-9)
    B = (np.log2(n_classes) + p * np.log2(p)
         + (1 - p) * np.log2((1 - p) / (n_classes - 1)))
    return float(max(0.0, B) * 60.0 / trial_sec)

class GigaDB22Dataset(Dataset):
    def __init__(self, h5_path):
        with h5py.File(h5_path, 'r') as f:
            eeg = f['eeg/epochs'][:]
            lbl = f['labels'][:].astype(np.int64) - 1
        eeg22 = eeg[:, GIGADB_22_IDX, GIGADB_MI_START:GIGADB_MI_END]
        self.eeg = torch.tensor(eeg22, dtype=torch.float32)
        self.lbl = torch.tensor(lbl,   dtype=torch.long)
    def __len__(self):  return len(self.lbl)
    def __getitem__(self, i): return self.eeg[i], self.lbl[i]

def bandpass_filter(data, low=4.0, high=40.0, fs=512, order=4):
    nyq = fs / 2.0
    b, a = butter(order, [low / nyq, high / nyq], btype='band')
    return filtfilt(b, a, data, axis=-1)

def load_bcic_subject(gdf_train, gdf_eval):
    all_eeg, all_lbl = [], []
    for gdf_path in [gdf_train, gdf_eval]:
        if not os.path.exists(gdf_path):
            print(f'  [SKIP] {gdf_path} not found'); continue
        with warnings.catch_warnings():
            warnings.simplefilter('ignore')
            raw = mne.io.read_raw_gdf(gdf_path, preload=True, verbose=False)
        eeg_picks = [i for i, ch in enumerate(raw.ch_names)
                     if not ch.startswith('EOG')][:22]
        data = raw.get_data(picks=eeg_picks)
        data = bandpass_filter(data, low=4.0, high=40.0, fs=FS_BCIC, order=4)
        events, eid = mne.events_from_annotations(raw, verbose=False)
        left_code  = eid.get(str(BCIC_LEFT_CODE),  None)
        right_code = eid.get(str(BCIC_RIGHT_CODE), None)
        if left_code is None or right_code is None:
            print(f'  [SKIP] {gdf_path}: cue 이벤트 없음'); continue
        target_codes = {left_code: 0, right_code: 1}
        for evt in events:
            onset, _, code = evt
            if code not in target_codes: continue
            end = onset + BCIC_EPOCH_SAMP
            if end > data.shape[1]: continue
            epoch = data[:, onset:end]
            epoch_512 = resample(epoch, N_TIMES, axis=-1)
            mu  = epoch_512.mean(axis=-1, keepdims=True)
            std = epoch_512.std(axis=-1, keepdims=True) + 1e-8
            epoch_512 = (epoch_512 - mu) / std
            all_eeg.append(epoch_512.astype(np.float32))
            all_lbl.append(target_codes[code])
    if not all_eeg: return None, None
    return np.stack(all_eeg), np.array(all_lbl, dtype=np.int64)

class BCICDataset(Dataset):
    def __init__(self, eeg, labels):
        self.eeg = torch.tensor(eeg,    dtype=torch.float32)
        self.lbl = torch.tensor(labels, dtype=torch.long)
    def __len__(self):  return len(self.lbl)
    def __getitem__(self, i): return self.eeg[i], self.lbl[i]

class EEGNet22(nn.Module):
    def __init__(self, n_ch=22, n_times=2048, F1=8, D=2,
                 kern_len=256, dropout=0.5, feat_dim=256, n_classes=2):
        super().__init__()
        F2 = F1 * D
        self.block1 = nn.Sequential(
            nn.Conv2d(1, F1, (1, kern_len), padding=(0, kern_len // 2), bias=False),
            nn.BatchNorm2d(F1),
        )
        self.block2 = nn.Sequential(
            nn.Conv2d(F1, F2, (n_ch, 1), groups=F1, bias=False),
            nn.BatchNorm2d(F2), nn.ELU(),
            nn.AvgPool2d((1, 4)), nn.Dropout(dropout),
        )
        self.block3 = nn.Sequential(
            nn.Conv2d(F2, F2, (1, 16), padding=(0, 8), groups=F2, bias=False),
            nn.Conv2d(F2, F2, 1, bias=False),
            nn.BatchNorm2d(F2), nn.ELU(),
            nn.AvgPool2d((1, 8)), nn.Dropout(dropout),
        )
        with torch.no_grad():
            flat = self.block3(self.block2(
                self.block1(torch.zeros(1, 1, n_ch, n_times)))).numel()
        self.clf = nn.Sequential(
            nn.Flatten(),
            nn.Linear(flat, feat_dim), nn.ELU(),
            nn.Linear(feat_dim, 128), nn.ELU(),
            nn.Dropout(dropout),
            nn.Linear(128, n_classes),
        )
    def forward(self, x):
        return self.clf(self.block3(self.block2(self.block1(x.unsqueeze(1)))))

def train_all_subjects(data_dir, ckpt_path, device, n_subjects=52,
                       max_epochs=100, patience=15):
    print(f'\n{"="*60}')
    print(f'  GigaDB 22ch EEG-only 학습 (all {n_subjects} subjects)')
    print(f'  Device: {device}')
    print(f'{"="*60}\n')
    all_ds = []
    for sid in range(1, n_subjects + 1):
        h5 = os.path.join(data_dir, f'sub-{sid:02d}_member_A.h5')
        if os.path.exists(h5): all_ds.append(GigaDB22Dataset(h5))
        else: print(f'  [SKIP] sub-{sid:02d} not found')
    full_ds = ConcatDataset(all_ds)
    n_total = len(full_ds)
    n_val   = max(int(n_total * 0.1), 64)
    n_train = n_total - n_val
    train_ds, val_ds = torch.utils.data.random_split(
        full_ds, [n_train, n_val],
        generator=torch.Generator().manual_seed(SEED))
    train_ldr = DataLoader(train_ds, batch_size=64, shuffle=True,
                           num_workers=2, pin_memory=True)
    val_ldr   = DataLoader(val_ds,   batch_size=64, shuffle=False, num_workers=0)
    model  = EEGNet22().to(device)
    optim  = torch.optim.Adam(model.parameters(), lr=1e-3, weight_decay=1e-4)
    sched  = torch.optim.lr_scheduler.CosineAnnealingLR(optim, T_max=max_epochs)
    scaler = _amp_scaler() if device.type == 'cuda' else None
    best_val_acc, patience_cnt, best_state = 0.0, 0, None
    for epoch in range(1, max_epochs + 1):
        model.train()
        for eeg, lbl in train_ldr:
            eeg, lbl = eeg.to(device), lbl.to(device)
            optim.zero_grad()
            with _amp_autocast(scaler is not None):
                loss = F.cross_entropy(model(eeg), lbl)
            if scaler:
                scaler.scale(loss).backward(); scaler.step(optim); scaler.update()
            else:
                loss.backward(); optim.step()
        sched.step()
        model.eval()
        preds, trues = [], []
        with torch.no_grad():
            for eeg, lbl in val_ldr:
                preds.extend(model(eeg.to(device)).argmax(1).cpu().tolist())
                trues.extend(lbl.tolist())
        val_acc = accuracy_score(trues, preds)
        if val_acc > best_val_acc:
            best_val_acc = val_acc; best_state = {k: v.clone() for k, v in model.state_dict().items()}; patience_cnt = 0
        else:
            patience_cnt += 1
            if patience_cnt >= patience:
                print(f'  Early stop @ epoch {epoch}, best val acc = {best_val_acc:.4f}'); break
        if epoch % 10 == 0:
            print(f'  Epoch {epoch:3d} | val_acc={val_acc:.4f} (best={best_val_acc:.4f}, patience={patience_cnt}/{patience})')
    model.load_state_dict(best_state)
    torch.save(best_state, ckpt_path)
    print(f'  모델 저장: {ckpt_path}')
    return model

@torch.no_grad()
def evaluate_bcic(model, bcic_dir, out_dir, device):
    print(f'\n{"="*60}')
    print('  BCIC IV 2a Zero-shot 평가')
    print(f'{"="*60}\n')
    model.eval()
    results = []
    for sid in range(1, 10):
        t_gdf = os.path.join(bcic_dir, f'A0{sid}T.gdf')
        e_gdf = os.path.join(bcic_dir, f'A0{sid}E.gdf')
        eeg, lbl = load_bcic_subject(t_gdf, e_gdf)
        if eeg is None: print(f'  [SKIP] A0{sid}: 데이터 로드 실패'); continue
        ds  = BCICDataset(eeg, lbl)
        ldr = DataLoader(ds, batch_size=64, shuffle=False)
        preds, trues = [], []
        for x, y in ldr:
            preds.extend(model(x.to(device)).argmax(1).cpu().tolist())
            trues.extend(y.tolist())
        pred_arr = np.array(preds); true_arr = np.array(trues)
        acc   = accuracy_score(true_arr, pred_arr)
        try:    kappa = cohen_kappa_score(true_arr, pred_arr, labels=[0, 1])
        except: kappa = 0.0
        itr = calc_itr(acc, trial_sec=4.0)
        cm  = confusion_matrix(true_arr, pred_arr, labels=[0, 1])
        lr  = cm[0,0]/cm[0].sum() if cm[0].sum() > 0 else 0.0
        rr  = cm[1,1]/cm[1].sum() if cm[1].sum() > 0 else 0.0
        r = dict(subject=f'A0{sid}', n_trials=len(trues),
                 accuracy=round(float(acc),6), kappa=round(float(kappa),6),
                 itr=round(itr,4), left_recall=round(float(lr),6), right_recall=round(float(rr),6))
        results.append(r)
        print(f'  A0{sid} | n={len(trues):3d} | acc={acc:.4f}  κ={kappa:.4f}  ITR={itr:.2f}')
    os.makedirs(out_dir, exist_ok=True)
    csv_path = os.path.join(out_dir, 'transfer_bcic2a_results.csv')
    pd.DataFrame(results).to_csv(csv_path, index=False)
    print(f'\n  결과 저장: {csv_path}')
    accs   = [r['accuracy'] for r in results]
    kappas = [r['kappa']    for r in results]
    itrs   = [r['itr']      for r in results]
    summary = {
        'n_subjects': len(results),
        'accuracy_mean': round(float(np.mean(accs)),4),
        'accuracy_std':  round(float(np.std(accs)),4),
        'kappa_mean':    round(float(np.mean(kappas)),4),
        'kappa_std':     round(float(np.std(kappas)),4),
        'itr_mean':      round(float(np.mean(itrs)),4),
        'itr_std':       round(float(np.std(itrs)),4),
        'transfer_config': {
            'source_dataset': 'GigaDB (Cho et al. 2017), 52 subjects',
            'target_dataset': 'BCI Competition IV 2a, 9 subjects',
            'n_channels': 22, 'channel_names': BCIC_22,
            'source_fs': FS_GIGADB, 'target_fs_orig': FS_BCIC,
            'target_fs_resampled': FS_TARGET, 'n_times': N_TIMES,
            'epoch_sec': 4.0, 'model': 'EEGNet22 (EEG-only, trained on all 52 GigaDB subjects)',
            'transfer_type': 'zero-shot (no fine-tuning on target data)',
        }
    }
    sum_path = os.path.join(out_dir, 'transfer_bcic2a_summary.json')
    with open(sum_path, 'w') as f: json.dump(summary, f, indent=2, ensure_ascii=False)
    print(f'  요약 저장: {sum_path}')
    print(f'\n  ── 요약 ──')
    print(f'  Accuracy : {np.mean(accs):.4f} ± {np.std(accs):.4f}')
    print(f'  Kappa    : {np.mean(kappas):.4f} ± {np.std(kappas):.4f}')
    print(f'  ITR      : {np.mean(itrs):.4f} ± {np.std(itrs):.4f}')
    return results

def parse_args():
    p = argparse.ArgumentParser(description='Zero-shot Transfer: GigaDB(22ch) → BCIC IV 2a',
                                formatter_class=argparse.ArgumentDefaultsHelpFormatter)
    p.add_argument('--data_dir',   default='BCI_Research/preprocessed/member_A')
    p.add_argument('--bcic_dir',   default='BCICIV_2a_gdf')
    p.add_argument('--out_dir',    default='BCI_Research/results/transfer_bcic2a')
    p.add_argument('--ckpt',       default=None)
    p.add_argument('--skip_train', action='store_true')
    p.add_argument('--drive_root', default=None)
    return p.parse_args()

def main():
    args   = parse_args()
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    if args.drive_root:
        root     = Path(args.drive_root)
        data_dir = args.data_dir or str(root / 'preprocessed' / 'member_A')
        bcic_dir = args.bcic_dir or str(root.parent / 'BCICIV_2a_gdf')
        out_dir  = args.out_dir  or str(root / 'results' / 'transfer_bcic2a')
    else:
        base     = Path(__file__).resolve().parent.parent
        data_dir = str(base / args.data_dir)
        bcic_dir = str(base / args.bcic_dir)
        out_dir  = str(base / args.out_dir)
    os.makedirs(out_dir, exist_ok=True)
    ckpt_path = args.ckpt or os.path.join(out_dir, 'transfer_22ch_model.pt')
    if args.skip_train and os.path.exists(ckpt_path):
        print(f'  체크포인트 로드: {ckpt_path}')
        model = EEGNet22().to(device)
        model.load_state_dict(torch.load(ckpt_path, map_location=device, weights_only=True))
    else:
        model = train_all_subjects(data_dir, ckpt_path, device)
    evaluate_bcic(model, bcic_dir, out_dir, device)

if __name__ == '__main__':
    main()
'''
        with open(_dst_file, 'w', encoding='utf-8') as _f:
            _f.write(_script_content)
        print(f'transfer_bcic2a.py → {_dst_file} 에 작성 완료')
    else:
        print(f'transfer_bcic2a.py 이미 존재: {_dst_file}')

if CODE_DIR not in sys.path:
    sys.path.insert(0, CODE_DIR)

script_path = os.path.join(CODE_DIR, _SCRIPT_NAME)
assert os.path.isfile(script_path), f'Script not found: {script_path}'
print(f'✅ Script found : {script_path}')
print(f'   CODE_DIR     : {CODE_DIR}')


transfer_bcic2a.py 이미 존재: /content/transfer_bcic2a.py
✅ Script found : /content/transfer_bcic2a.py
   CODE_DIR     : /content


---
## Step 5 — Run the transfer experiment

This cell trains `EEGNet22` on all 52 GigaDB subjects (22-channel subset) and then evaluates zero-shot on 9 BCIC IV 2a subjects.

**Estimated time on T4 GPU:** 15–25 min for training, ~2 min for evaluation.

> If you already have `transfer_22ch_model.pt` in `OUT_DIR`, change `SKIP_TRAIN = False` → `True` below to skip training.

In [25]:
SKIP_TRAIN = False   # set True if transfer_22ch_model.pt already exists

skip_flag = '--skip_train' if SKIP_TRAIN else ''

cmd = (
    f'python "{script_path}" '
    f'--data_dir "{DATA_DIR}" '
    f'--bcic_dir "{BCIC_DIR}" '
    f'--out_dir  "{OUT_DIR}" '
    f'--drive_root "{DRIVE_ROOT}" '
    f'{skip_flag}'
)

print('Running command:')
print(cmd)
print('─' * 60)

import subprocess, sys
result = subprocess.run(cmd, shell=True, capture_output=False, text=True)
if result.returncode != 0:
    print('\n❌ Script exited with non-zero return code:', result.returncode)
else:
    print('\n✅ Script completed successfully.')

Running command:
python "/content/transfer_bcic2a.py" --data_dir "/content/drive/MyDrive/BCI_Research/preprocessed/member_A" --bcic_dir "/content/drive/MyDrive/BCI_Research/BCICIV_2a_gdf" --out_dir  "/content/drive/MyDrive/BCI_Research/results/transfer_bcic2a" --drive_root "/content/drive/MyDrive/BCI_Research" 
────────────────────────────────────────────────────────────

✅ Script completed successfully.


---
## Step 6 — Load and display results

In [26]:
import pandas as pd, json, os

csv_path  = os.path.join(OUT_DIR, 'transfer_bcic2a_results.csv')
json_path = os.path.join(OUT_DIR, 'transfer_bcic2a_summary.json')

assert os.path.isfile(csv_path),  f'Results CSV not found: {csv_path}'
assert os.path.isfile(json_path), f'Summary JSON not found: {json_path}'

df = pd.read_csv(csv_path)
with open(json_path) as f:
    summary = json.load(f)

print('=== Per-Subject Results ===')
print(df.to_string(index=False))
print()
print('=== Summary (mean ± SD) ===')
for k, v in summary.items():
    print(f'  {k}: {v}')

=== Per-Subject Results ===
subject  n_trials  accuracy     kappa    itr  left_recall  right_recall
    A01       144  0.520833  0.041667 0.0188     0.041667      1.000000
    A02       144  0.486111 -0.027778 0.0084     0.027778      0.944444
    A03       144  0.506944  0.013889 0.0021     0.097222      0.916667
    A04       144  0.479167 -0.041667 0.0188     0.027778      0.930556
    A05       144  0.500000  0.000000 0.0000     0.027778      0.972222
    A06       144  0.506944  0.013889 0.0021     0.486111      0.527778
    A07       144  0.493056 -0.013889 0.0021     0.000000      0.986111
    A08       144  0.472222 -0.055556 0.0334     0.055556      0.888889
    A09       144  0.513889  0.027778 0.0084     0.194444      0.833333

=== Summary (mean ± SD) ===
  n_subjects: 9
  accuracy_mean: 0.4977
  accuracy_std: 0.0154
  kappa_mean: -0.0046
  kappa_std: 0.0307
  itr_mean: 0.0105
  itr_std: 0.0105
  transfer_config: {'source_dataset': 'GigaDB (Cho et al. 2017), 52 subjects', 't

---
## Step 7 — Format table for §3.9 of results_draft.md

In [27]:
import pandas as pd, json

# Build markdown table
header = '| Subject | Acc (%) | κ | ITR (bits/min) |'
sep    = '|---------|---------|---|----------------|'
rows   = [header, sep]

for _, row in df.iterrows():
    subj = row.get('subject', row.get('Subject', '?'))
    acc  = row.get('accuracy', row.get('Accuracy', 0)) * 100
    kappa = row.get('kappa', row.get('Cohen_kappa', 0))
    itr  = row.get('itr', row.get('ITR', 0))
    rows.append(f'| {subj} | {acc:.1f} | {kappa:.3f} | {itr:.3f} |')

# Mean ± SD footer
mean_acc  = summary.get('accuracy_mean', summary.get('mean_accuracy', 0)) * 100
std_acc   = summary.get('accuracy_std',  summary.get('std_accuracy', 0)) * 100
mean_k    = summary.get('kappa_mean', summary.get('mean_kappa', 0))
std_k     = summary.get('kappa_std',  summary.get('std_kappa', 0))
mean_itr  = summary.get('itr_mean', summary.get('mean_itr', 0))
std_itr   = summary.get('itr_std',  summary.get('std_itr', 0))
rows.append(f'| **Mean ± SD** | **{mean_acc:.1f} ± {std_acc:.1f}** | **{mean_k:.3f} ± {std_k:.3f}** | **{mean_itr:.3f} ± {std_itr:.3f}** |')

table_md = '\n'.join(rows)
print('=== Copy this into results_draft.md §3.9.1 ===')
print()
print(table_md)
print()
print('=== Benchmark comparison row (fill into §3.9.2 table) ===')
print(f'| EEGNet-22ch (ours, zero-shot) | Cross-dataset, 2-class | **{mean_acc:.1f}** | No subject adaptation |')

=== Copy this into results_draft.md §3.9.1 ===

| Subject | Acc (%) | κ | ITR (bits/min) |
|---------|---------|---|----------------|
| A01 | 52.1 | 0.042 | 0.019 |
| A02 | 48.6 | -0.028 | 0.008 |
| A03 | 50.7 | 0.014 | 0.002 |
| A04 | 47.9 | -0.042 | 0.019 |
| A05 | 50.0 | 0.000 | 0.000 |
| A06 | 50.7 | 0.014 | 0.002 |
| A07 | 49.3 | -0.014 | 0.002 |
| A08 | 47.2 | -0.056 | 0.033 |
| A09 | 51.4 | 0.028 | 0.008 |
| **Mean ± SD** | **0.0 ± 0.0** | **0.000 ± 0.000** | **0.000 ± 0.000** |

=== Benchmark comparison row (fill into §3.9.2 table) ===
| EEGNet-22ch (ours, zero-shot) | Cross-dataset, 2-class | **0.0** | No subject adaptation |


---
## Step 8 — Sanity checks

Run these cells to validate the results before copying them into the paper.

In [28]:
import numpy as np

accs = df['accuracy'].values if 'accuracy' in df.columns else df['Accuracy'].values

# 1. Check for chance-level or below-chance performance
below_chance = np.sum(accs <= 0.50)
print(f'Subjects at or below chance (≤50%): {below_chance}/9')
if below_chance > 0:
    print('  ⚠️  Warning: check label alignment in load_bcic_subject() — event codes 769/770 ↔ class 0/1 mapping')

# 2. Check mean accuracy vs. naive expectation
mean_acc = float(np.mean(accs))
print(f'Mean accuracy: {mean_acc*100:.1f}%')
if mean_acc < 0.55:
    print('  ⚠️  Below 55% — may indicate label flip or preprocessing issue')
elif mean_acc > 0.65:
    print('  ✅  Above 65% — reasonable zero-shot transfer')
else:
    print('  ℹ️  55–65% — marginal transfer; discuss domain gap in §3.9.3')

# 3. Variance
std_acc = float(np.std(accs))
print(f'Std dev: {std_acc*100:.1f}%')
if std_acc > 0.15:
    print('  ℹ️  High variance — consider noting inter-subject variability in §3.9.3')

print('\n✅ Sanity checks complete.')

Subjects at or below chance (≤50%): 5/9
  ⚠️  Warning: check label alignment in load_bcic_subject() — event codes 769/770 ↔ class 0/1 mapping
Mean accuracy: 49.8%
  ⚠️  Below 55% — may indicate label flip or preprocessing issue
Std dev: 1.5%

✅ Sanity checks complete.


---
## Notes for paper write-up (§3.9.3)

After running and checking results, update `results_draft.md §3.9` as follows:

1. **§3.9.1 table** — replace all `[TODO]` values with the per-subject numbers from Step 7.
2. **§3.9.2 benchmark row** — replace `[TODO]` in the EEGNet-22ch row with the mean accuracy.
3. **§3.9.3 interpretation** — fill in the blanks using the template:
   > "Zero-shot cross-dataset accuracy of **X.X ± Y.Y%** (κ = **Z.ZZZ**) on BCIC IV 2a demonstrates [adequate / limited] generalisation of the GigaDB-trained EEG encoder across acquisition conditions. The performance gap relative to within-subject EEGNet baseline (ΔAcc ≈ **PP pp**) is expected given the absence of subject-specific adaptation and the domain shift in channel configuration, sampling rate, and electrode cap. These results [support / motivate] future work on domain-adaptive transfer learning approaches (§4.7)."
4. Also update `abstract_draft.md` and `conclusion_draft.md` to include the §3.9 finding in the results summary and contributions list respectively.

**File locations:**
- Results CSV: `BCI_Research/results/transfer_bcic2a/transfer_bcic2a_results.csv`
- Summary JSON: `BCI_Research/results/transfer_bcic2a/transfer_bcic2a_summary.json`
- Paper drafts: `BCI_Research/code/MI-BCI/paper/`